# 🔑 Solutions: Lesson 08 · Convolutional Neural Networks

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 8.1: Output size calculator  &nbsp; `⭐`

Write `conv_out(n, k, s=1, p=0)` returning the output width/height of a convolution or pooling layer. Then use it to compute the final spatial size of: 64×64 input → conv(k=3,s=1,p=1) → pool(k=2,s=2) → conv(k=3,s=1,p=0) → pool(k=2,s=2), storing it in `final_size`.


In [2]:
def conv_out(n, k, s=1, p=0):
    return (n + 2 * p - k) // s + 1

n = 64
n = conv_out(n, 3, 1, 1)
n = conv_out(n, 2, 2, 0)
n = conv_out(n, 3, 1, 0)
n = conv_out(n, 2, 2, 0)
final_size = n

In [3]:
# ▶ self-check
dl.check("conv_out(28,5,1,0) = 24", conv_out(28, 5, 1, 0) == 24)
dl.check("conv_out(28,5,1,2) = 28 ('same' padding)", conv_out(28, 5, 1, 2) == 28)
dl.check("conv_out(28,3,2,1) = 14", conv_out(28, 3, 2, 1) == 14)
dl.check("final_size = 15", final_size == 15, "trace each step on paper")

✅ conv_out(28,5,1,0) = 24
✅ conv_out(28,5,1,2) = 28 ('same' padding)
✅ conv_out(28,3,2,1) = 14
✅ final_size = 15


**Why this works.** 64 →(conv p=1) 64 →(pool) 32 →(conv, no pad) 30 →(pool) 15.

---
### 🧩 Problem 8.2: Count conv parameters  &nbsp; `⭐`

Write `conv_params(c_in, c_out, k, bias=True)`. Then compute `total` = parameters of: Conv(3→16,k=3) + Conv(16→32,k=3) + Conv(32→64,k=3).


In [4]:
def conv_params(c_in, c_out, k, bias=True):
    return k * k * c_in * c_out + (c_out if bias else 0)

total = conv_params(3, 16, 3) + conv_params(16, 32, 3) + conv_params(32, 64, 3)

In [5]:
# ▶ self-check
dl.check("conv_params(1,16,5) = 416", conv_params(1, 16, 5) == 416)
dl.check("without bias: 400", conv_params(1, 16, 5, bias=False) == 400)
dl.check("total = 448 + 4640 + 18496 = 23,584", total == 23584)

✅ conv_params(1,16,5) = 416
✅ without bias: 400
✅ total = 448 + 4640 + 18496 = 23,584


---
### 🧩 Problem 8.3: Implement convolution (with stride & padding)  &nbsp; `⭐⭐⭐`

Write `conv2d(x, k, stride=1, padding=0)` for a 2-D single-channel image `x` and kernel `k` (cross-correlation, zero padding). The self-check compares with PyTorch's `F.conv2d`.


In [6]:
def conv2d(x, k, stride=1, padding=0):
    if padding:
        x = np.pad(x, padding)
    H, W = x.shape; kh, kw = k.shape
    oh, ow = (H - kh) // stride + 1, (W - kw) // stride + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = np.sum(x[i*stride:i*stride+kh, j*stride:j*stride+kw] * k)
    return out

In [7]:
# ▶ self-check
import torch, torch.nn.functional as F
rng = np.random.default_rng(1); x = rng.normal(size=(9, 9)); k = rng.normal(size=(3, 3))
for s, p in [(1, 0), (1, 1), (2, 1)]:
    mine = conv2d(x, k, s, p)
    ref = F.conv2d(torch.tensor(x)[None, None], torch.tensor(k)[None, None], stride=s, padding=p)[0, 0].numpy()
    dl.check(f"stride={s}, padding={p}", mine is not None and mine.shape == ref.shape and np.allclose(mine, ref))

✅ stride=1, padding=0
✅ stride=1, padding=1
✅ stride=2, padding=1


---
### 🧩 Problem 8.4: Max pooling for any window size  &nbsp; `⭐⭐`

Write `max_pool(x, size)` (stride = size) for a 2-D array whose sides are divisible by `size`. Use the reshape trick: `x.reshape(H//size, size, W//size, size).max(axis=(1, 3))`.


In [8]:
def max_pool(x, size=2):
    H, W = x.shape
    return x.reshape(H // size, size, W // size, size).max(axis=(1, 3))

In [9]:
# ▶ self-check
x = np.arange(16, dtype=float).reshape(4, 4)
dl.check("2×2 pooling gives [[5,7],[13,15]]", max_pool(x, 2) is not None and np.array_equal(max_pool(x, 2), [[5, 7], [13, 15]]))
dl.check("4×4 pooling gives one number: 15", max_pool(x, 4) is not None and np.array_equal(max_pool(x, 4), [[15]]))

✅ 2×2 pooling gives [[5,7],[13,15]]
✅ 4×4 pooling gives one number: 15


---
### 🧩 Problem 8.5: Predict the flatten size  &nbsp; `⭐⭐`

Write `flat_features(input_shape, layers)` where `input_shape=(C,H,W)` and `layers` is a list like
`[("conv", c_out, k, s, p), ("pool", k)]` (pool stride = k). Return the number of features after flattening.
This is exactly the number you must put in the first `nn.Linear` after `nn.Flatten()`. Verify on the leaf-classifier CNN below.


In [10]:
def flat_features(input_shape, layers):
    C, H, W = input_shape
    for layer in layers:
        if layer[0] == "conv":
            _, c_out, k, s, p = layer
            C, H, W = c_out, (H + 2 * p - k) // s + 1, (W + 2 * p - k) // s + 1
        else:
            k = layer[1]; H, W = H // k, W // k
    return C * H * W

In [11]:
# ▶ self-check
mnist = [("conv", 8, 5, 1, 2), ("pool", 2), ("conv", 16, 5, 1, 2), ("pool", 2)]
dl.check("MNIST CNN flattens to 784", flat_features((1, 28, 28), mnist) == 784)
leaf = [("conv", 16, 3, 1, 1), ("pool", 2), ("conv", 32, 3, 1, 1), ("pool", 2), ("conv", 64, 3, 1, 1), ("pool", 2)]
dl.check("leaf CNN (48×48 RGB) flattens to 64·6·6 = 2304", flat_features((3, 48, 48), leaf) == 2304)

✅ MNIST CNN flattens to 784
✅ leaf CNN (48×48 RGB) flattens to 64·6·6 = 2304
